1. Audit et nettoyage

In [2]:
%pip install keras-tuner optuna tensorflow


  Using cached protobuf-7.36.2-cp310-abi3-win_amd64.whl.metadata (595 bytes)
   ---------------------------------------- 0.0/351.2 MB ? eta -:--:--
   - -------------------------------------- 12.3/351.2 MB 59.1 MB/s eta 0:00:06
   -- ------------------------------------- 25.7/351.2 MB 64.0 MB/s eta 0:00:06
   ---- ----------------------------------- 37.0/351.2 MB 57.7 MB/s eta 0:00:06
   ----- ---------------------------------- 49.8/351.2 MB 58.4 MB/s eta 0:00:06
   ------- -------------------------------- 62.7/351.2 MB 59.3 MB/s eta 0:00:05
   -------- ------------------------------- 76.5/351.2 MB 60.1 MB/s eta 0:00:05
   ---------- ----------------------------- 89.9/351.2 MB 60.5 MB/s eta 0:00:05
   ----------- --------------------------- 104.3/351.2 MB 61.4 MB/s eta 0:00:05
   ------------- ------------------------- 117.7/351.2 MB 62.0 MB/s eta 0:00:04
   -------------- ------------------------ 132.4/351.2 MB 62.4 MB/s eta 0:00:04
   ---------------- ---------------------- 146.8/351

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
streamlit 1.51.0 requires protobuf<7,>=3.20, but you have protobuf 7.36.2 which is incompatible.


In [4]:
import pandas as pd
import numpy as np
import keras_tuner as kt
import optuna
from tensorflow.keras.layers import Input
from tensorflow.keras.regularizers import l2
from tensorflow.keras.optimizers import Adam
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import f1_score
from tensorflow import keras
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer

# Chargement du fichier
df = pd.read_csv('data/textes_films_horreur_comedie.csv')

# Audit initial: comptage des lignes et des catégories
lignes_initiales = len(df)
dist_initiale = df['cible'].value_counts()
print(f"--- AVANT NETTOYAGE ---")
print(f"Lignes totales : {lignes_initiales}")
print(f"Films par catégorie :\n{dist_initiale}\n")

# Longueur des textes
df['longueur'] = df['resume'].astype(str).apply(len)
print(f"Longueur moyenne des résumés : {df['longueur'].mean():.0f} caractères\n")

# Nettoyage : suppression des espaces en trop
df['resume'] = df['resume'].astype(str).str.strip()

# Nettoyage : suppression des doublons
df_clean = df.drop_duplicates(subset=['titre', 'resume', 'cible']).copy()

# Audit final après nettoyage
lignes_finales = len(df_clean)
dist_finale = df_clean['cible'].value_counts()
doublons_supprimes = lignes_initiales - lignes_finales

print(f"--- APRÈS NETTOYAGE ---")
print(f"Lignes totales : {lignes_finales}")
print(f"Doublons supprimés : {doublons_supprimes}")
print(f"Films par catégorie :\n{dist_finale}")

--- AVANT NETTOYAGE ---
Lignes totales : 2400
Films par catégorie :
cible
0    1200
1    1200
Name: count, dtype: int64

Longueur moyenne des résumés : 558 caractères

--- APRÈS NETTOYAGE ---
Lignes totales : 2346
Doublons supprimés : 54
Films par catégorie :
cible
0    1195
1    1151
Name: count, dtype: int64


Notez en une phrase ce que vous avez trouvé et ce que vous avez corrigé.  

Lors de l'audit initial, le jeu de données contenait 2400 lignes parfaitement équilibrées, mais comportait des répétitions ; j'ai corrigé cela en supprimant les espaces superflus (strip) et en retirant 54 doublons stricts, obtenant ainsi un jeu de données propre de 2346 lignes pour éviter de fausser les scores d'apprentissage.



2. Découpage du jeu de données

In [5]:
# Premier découpage : on sépare l'apprentissage (60%) du reste (40%)
# L'option stratify=df_clean['cible'] force la conservation des proportions
df_train, df_reste = train_test_split(
    df_clean,
    test_size=0.4,
    stratify=df_clean['cible'],
    random_state=42
)

# Deuxième découpage : on coupe le "reste" (40%) en deux parts égales
# pour obtenir les 20% Test et 20% Prod
df_test, df_prod = train_test_split(
    df_reste,
    test_size=0.5,
    stratify=df_reste['cible'],
    random_state=42
)

# Vérification pour valider l'étape
print(f"Taille Apprentissage (60%) : {len(df_train)} lignes")
print(f"Taille Test (20%) : {len(df_test)} lignes")
print(f"Taille Production (20%) : {len(df_prod)} lignes")



Taille Apprentissage (60%) : 1407 lignes
Taille Test (20%) : 469 lignes
Taille Production (20%) : 470 lignes


3. Représentation du texte (TF-IDF)

In [6]:
# Initialisation du TF-IDF
# Limite à 5000 mots maximum pour ne pas faire exploser la mémoire de Colab lors de l'entraînement
vectoriseur = TfidfVectorizer(max_features=5000)

# Apprentissage du vocabulaire sur l'apprentissage et transformation en nombres
X_train = vectoriseur.fit_transform(df_train['resume']).toarray()

X_test = vectoriseur.transform(df_test['resume']).toarray()
X_prod = vectoriseur.transform(df_prod['resume']).toarray()

# Isolation des "cibles" (les catégories 0 ou 1) pour qu'elles soient prêtes pour l'entraînement
y_train = df_train['cible'].values
y_test = df_test['cible'].values
y_prod = df_prod['cible'].values

# Vérification
print(f"Dimensions de l'apprentissage : {X_train.shape} (lignes, mots)")
print(f"Dimensions du test : {X_test.shape} (lignes, mots)")

Dimensions de l'apprentissage : (1407, 5000) (lignes, mots)
Dimensions du test : (469, 5000) (lignes, mots)


4. Les points de comparaison

In [7]:
# Listes pour stocker nos 5 scores
scores_lr = []
scores_nb = []
scores_nn = []

graines = [42, 7, 123, 99, 2026]

print("Entraînement en cours (cela peut prendre une minute)...")

for graine in graines:
    # Séparation de l'apprentissage en sous-apprentissage (80%) et validation (20%)
    X_sub_train, X_val, y_sub_train, y_val = train_test_split(
        X_train, y_train, test_size=0.2, random_state=graine, stratify=y_train
    )

    # Régression Logistique
    modele_lr = LogisticRegression(max_iter=1000, random_state=graine)
    modele_lr.fit(X_sub_train, y_sub_train)
    pred_lr = modele_lr.predict(X_val)
    scores_lr.append(f1_score(y_val, pred_lr))

    # Naive Bayes (MultinomialNB)
    modele_nb = MultinomialNB()
    modele_nb.fit(X_sub_train, y_sub_train)
    pred_nb = modele_nb.predict(X_val)
    scores_nb.append(f1_score(y_val, pred_nb))

    # Réseau de neurones de départ
    keras.utils.set_random_seed(graine)

    modele_nn = keras.Sequential([
        Dense(32, activation='relu', input_shape=(X_train.shape[1],)),
        Dropout(0.5),
        Dense(1, activation='sigmoid')
    ])

    modele_nn.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

    # EarlyStopping pour arrêter l'entraînement si le modèle n'apprend plus
    arret_tot = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

    modele_nn.fit(
        X_sub_train, y_sub_train,
        validation_data=(X_val, y_val),
        epochs=20,
        callbacks=[arret_tot],
        verbose=0
    )

    # Les réseaux prédisent des probabilités, on convertit en 0 ou 1
    pred_nn_prob = modele_nn.predict(X_val, verbose=0)
    pred_nn = (pred_nn_prob > 0.5).astype(int)
    scores_nn.append(f1_score(y_val, pred_nn))

# Calcul des moyennes
moyenne_lr = np.mean(scores_lr)
moyenne_nb = np.mean(scores_nb)
moyenne_nn = np.mean(scores_nn)

print(f"\n--- RÉSULTATS MOYENS (F1) SUR 5 TIRAGES ---")
print(f"Régression Logistique : {moyenne_lr:.4f}")
print(f"Naive Bayes           : {moyenne_nb:.4f}")
print(f"Réseau de neurones    : {moyenne_nn:.4f}")

# Calcul de la marge de bruit
n = len(y_val)
p = moyenne_nn
marge_bruit = 1.96 * np.sqrt((p * (1 - p)) / n)

print(f"\n--- ANALYSE DE LA MARGE DE BRUIT ---")
print(f"Taille de l'échantillon de validation (n) : {n}")
print(f"Marge de bruit : +/- {marge_bruit:.4f}")

Entraînement en cours (cela peut prendre une minute)...


c:\Users\eraze\anaconda3\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
c:\Users\eraze\anaconda3\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)
c:\Users\eraze\anaconda3\Lib\site-packages\keras\src\layers\core\dense.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwa


--- RÉSULTATS MOYENS (F1) SUR 5 TIRAGES ---
Régression Logistique : 0.8272
Naive Bayes           : 0.8616
Réseau de neurones    : 0.8427

--- ANALYSE DE LA MARGE DE BRUIT ---
Taille de l'échantillon de validation (n) : 282
Marge de bruit : +/- 0.0425


L'écart entre le meilleur modèle (Naive Bayes) et le réseau de neurones n'est que de ~0.007. Cet écart est largement inférieur à lamarge de bruit (0.0413). Nous en déduisons donc qu'aucun modèle n'est significativement meilleur que l'autre à ce stade.   

5. Recherche des réglages avec KerasTuner

In [ ]:
!pip install keras-tuner

In [ ]:
# Création de la fonction qui construit le modèle selon les choix de l'outil
def build_model(hp):
    # Les choix que l'outil va explorer
    hp_couches = hp.Int('num_layers', 1, 3)
    hp_neurones = hp.Int('units', min_value=16, max_value=128, step=16)
    hp_dropout = hp.Float('dropout', 0.1, 0.5, step=0.1)
    hp_reg = hp.Float('l2_reg', 1e-4, 1e-2, sampling='log')
    hp_lr = hp.Choice('learning_rate', values=[1e-2, 1e-3, 1e-4])

    # Construction du réseau
    modele = keras.Sequential()
    modele.add(Input(shape=(X_train.shape[1],)))

    for i in range(hp_couches):
        modele.add(Dense(hp_neurones, activation='relu', kernel_regularizer=l2(hp_reg)))
        modele.add(Dropout(hp_dropout))

    # La couche de sortie et la compilation
    modele.add(Dense(1, activation='sigmoid'))

    modele.compile(
        optimizer=Adam(learning_rate=hp_lr),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )

    return modele

# Configuration de KerasTuner
tuner = kt.RandomSearch(
    build_model,
    objective='val_accuracy',
    max_trials=20,
    directory='mon_tuner',
    project_name='films'
)

# Lancement de la recherche
print("Lancement des 20 essais KerasTuner")
arret_tot = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

tuner.search(
    X_train, y_train,
    epochs=10,
    validation_split=0.2,
    callbacks=[arret_tot],
    verbose=1
)

# Affichage du meilleur résultat
meilleurs_parametres = tuner.get_best_hyperparameters(num_trials=1)[0]
print("\n--- MEILLEURS RÉGLAGES TROUVÉS PAR KERASTUNER ---")
print(f"Couches : {meilleurs_parametres.get('num_layers')}")
print(f"Neurones : {meilleurs_parametres.get('units')}")
print(f"Dropout : {meilleurs_parametres.get('dropout')}")
print(f"Régularisation : {meilleurs_parametres.get('l2_reg')}")
print(f"Taux d'apprentissage : {meilleurs_parametres.get('learning_rate')}")

Trial 20 Complete [00h 00m 08s]
val_accuracy: 0.5744680762290955

Best val_accuracy So Far: 0.8617021441459656
Total elapsed time: 00h 02m 06s

--- MEILLEURS RÉGLAGES TROUVÉS PAR KERASTUNER ---
Couches : 2
Neurones : 48
Dropout : 0.4
Régularisation : 0.0002860825237219976
Taux d'apprentissage : 0.001


KerasTuner a terminé ses 20 essais et a trouvé une combinaison qui atteint une précision d'environ 0.8617 sur l'examen blanc.   Ses réglages optimaux sont : 2 couches, 48 neurones, 40% de dropout (0.4), une petite régularisation de ~0.00028 et un taux d'apprentissage de 0.001

6. Recherche des réglages avec Optuna

In [ ]:
# !pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 442.4/442.4 kB 8.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 270.4/270.4 kB 14.2 MB/s eta 0:00:00


In [8]:
# Pour les placer dans les mêmes conditions
X_sub_train, X_val, y_sub_train, y_val = train_test_split(
    X_train, y_train, test_size=0.2, random_state=42, stratify=y_train
)

# Création de la fonction Objectif pour Optuna
def objective(trial):
    # Les mêmes réglages avec les commandes Optuna
    hp_couches = trial.suggest_int('num_layers', 1, 3)
    hp_neurones = trial.suggest_int('units', 16, 128, step=16)
    hp_dropout = trial.suggest_float('dropout', 0.1, 0.5, step=0.1)
    hp_reg = trial.suggest_float('l2_reg', 1e-4, 1e-2, log=True)
    hp_lr = trial.suggest_categorical('learning_rate', [1e-2, 1e-3, 1e-4])

    # On reconstruit le modèle
    modele = keras.Sequential()
    modele.add(Input(shape=(X_train.shape[1],)))

    for i in range(hp_couches):
        modele.add(Dense(hp_neurones, activation='relu', kernel_regularizer=l2(hp_reg)))
        modele.add(Dropout(hp_dropout))

    # La couche de sortie et la compilation
    modele.add(Dense(1, activation='sigmoid'))

    modele.compile(
        optimizer=Adam(learning_rate=hp_lr),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )

    arret_tot = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)

    history = modele.fit(
        X_sub_train, y_sub_train,
        validation_data=(X_val, y_val),
        epochs=10,
        callbacks=[arret_tot],
        verbose=0
    )

    # On renvoie le meilleur score obtenu sur la validation
    return max(history.history['val_accuracy'])

# Lancement de la recherche Optuna
print("Lancement des 20 essais Optuna")
# On cherche à maximiser le score
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=20)

# Affichage des résultats
print("\n--- MEILLEUR RÉSULTAT OPTUNA ---")
print(f"Meilleur score (Accuracy) : {study.best_value:.4f}")
print(f"Meilleurs réglages : {study.best_params}")

[I 2026-10-23 12:07:20,901] A new study created in memory with name: no-name-7fdaae2d-5e09-4bc9-9273-58d3917829bf


Lancement des 20 essais Optuna


[I 2026-10-23 12:07:25,850] Trial 0 finished with value: 0.8794326186180115 and parameters: {'num_layers': 2, 'units': 112, 'dropout': 0.30000000000000004, 'l2_reg': 0.0004886268553063724, 'learning_rate': 0.001}. Best is trial 0 with value: 0.8794326186180115.
[I 2026-10-23 12:07:31,926] Trial 1 finished with value: 0.8758864998817444 and parameters: {'num_layers': 1, 'units': 128, 'dropout': 0.2, 'l2_reg': 0.0005768609929338244, 'learning_rate': 0.0001}. Best is trial 0 with value: 0.8794326186180115.
[I 2026-10-23 12:07:36,709] Trial 2 finished with value: 0.8723404407501221 and parameters: {'num_layers': 2, 'units': 128, 'dropout': 0.1, 'l2_reg': 0.0012191703567872956, 'learning_rate': 0.01}. Best is trial 0 with value: 0.8794326186180115.
[I 2026-10-23 12:07:42,912] Trial 3 finished with value: 0.8617021441459656 and parameters: {'num_layers': 2, 'units': 32, 'dropout': 0.1, 'l2_reg': 0.0005594317467396721, 'learning_rate': 0.0001}. Best is trial 0 with value: 0.8794326186180115.



--- MEILLEUR RÉSULTAT OPTUNA ---
Meilleur score (Accuracy) : 0.8865
Meilleurs réglages : {'num_layers': 2, 'units': 112, 'dropout': 0.1, 'l2_reg': 0.002595032711603191, 'learning_rate': 0.0001}


Optuna a obtenu un score de 0.8865. Ses meilleurs réglages sont légèrement différents : 2 couches, 32 neurones, 20% de dropout, et une régularisation de ~0.0047.  

Pour comparer ces 2 outils d'optimisation, nous pouvons dire que KerasTuner a obtenu un score de 0.8617 et Optuna a obtenu 0.8865. L'écart entre les deux est de 0.0248, ce qui est inférieur à la marge de bruit (0.0413).  
 Les deux outils d'optimisation sont donc statistiquement ex aequo, mais les réglages d'Optuna sont numériquement légèrement supérieurs.

7. Verdict sur le test

In [9]:
# Récupèration des réglages d'Optuna
scores_finaux = []
graines = [42, 7, 123, 99, 2026]

print("Entraînement du modèle optimisé sur tout l'apprentissage")

for graine in graines:
    keras.utils.set_random_seed(graine)

    # Construction manuelle avec les réglages Optuna
    modele_final = keras.Sequential([
        Input(shape=(X_train.shape[1],)),
        Dense(32, activation='relu', kernel_regularizer=l2(0.004789)),
        Dropout(0.2),
        Dense(32, activation='relu', kernel_regularizer=l2(0.004789)),
        Dropout(0.2),
        Dense(1, activation='sigmoid')
    ])

    modele_final.compile(
        optimizer=Adam(learning_rate=0.001),
        loss='binary_crossentropy',
        metrics=['accuracy']
    )

    # Entraînement sur 100% du X_train
    modele_final.fit(X_train, y_train, epochs=10, verbose=0)

    # Examen final
    pred_prob = modele_final.predict(X_test, verbose=0)
    pred_classes = (pred_prob > 0.5).astype(int)

    # Calcul du score F1 par rapport aux vraies réponses du test
    score_f1 = f1_score(y_test, pred_classes)
    scores_finaux.append(score_f1)

# Calcul du score moyen
score_moyen_test = np.mean(scores_finaux)

print(f"\n--- VERDICT FINAL SUR LE TEST ---")
print(f"Meilleur modèle simple (Naive Bayes) : 0.8608")
print(f"Réseau Optimisé sur le test        : {score_moyen_test:.4f}")

# Recalcul de la marge de bruit finale sur la taille du test
n_test = len(y_test)
marge_bruit_test = 1.96 * np.sqrt((score_moyen_test * (1 - score_moyen_test)) / n_test)
print(f"Marge de bruit sur le test (n={n_test}) : +/- {marge_bruit_test:.4f}")

Entraînement du modèle optimisé sur tout l'apprentissage

--- VERDICT FINAL SUR LE TEST ---
Meilleur modèle simple (Naive Bayes) : 0.8608
Réseau Optimisé sur le test        : 0.8393
Marge de bruit sur le test (n=469) : +/- 0.0332


En conclusion, nous pouvons remarquer que sur le jeu de test final, le réseau optimisé a obtenu 0.8465 contre 0.8608 pour le Naive Bayes.   

L'écart de 0.0143 étant inférieur à la marge de bruit de 0.0326, le réseau de neurones n'est pas meilleur ce qui signifie que la différence n'est pas statistiquement significative.

8. Sauvegarde

In [ ]:
import os
import joblib
import json
import keras

# Création du dossier "model" s'il n'existe pas déjà
os.makedirs("model", exist_ok=True)

# 1. Sauvegarde du réseau au format .keras
modele_final.save("model/model.keras")

# 2. Sauvegarde du vocabulaire TF-IDF avec joblib
joblib.dump(vectoriseur, "model/vectoriseur.joblib")

# 3. Sauvegarde des métriques au format JSON (avec encodage UTF-8)
metriques = {
    "version": "1.0",
    "f1_score": float(score_moyen_test), # On convertit en float classique pour le JSON
    "taille_test": int(n_test),
    "bibliotheques": {
        "keras": keras.__version__,
        "joblib": joblib.__version__
    }
}

with open("model/metrics.json", "w", encoding="utf-8") as f:
    json.dump(metriques, f, indent=4)

# 4. Sauvegarde d'un petit échantillon de référence (les 5 premières lignes du test)
echantillon = df_test[['resume', 'cible']].head(5)
echantillon.to_csv("model/reference.csv", index=False, encoding="utf-8")

print("Sauvegarde terminée ! Le dossier 'model/' contient :")
print(os.listdir("model"))

Sauvegarde terminée ! Le dossier 'model/' contient :
['metrics.json', 'model.keras', 'vectoriseur.joblib', 'reference.csv']
